# Credit Risk Portfolio Analysis

## Project aim

Lending companies cannot manually review every loan application in the same level of detail. In this project, I used historical Lending Club data to investigate whether higher-risk loans could be identified and placed earlier in a manual-review queue.

I compared review capacities of 5%, 10% and 20%. For each level, I measured the number of defaults found, the value of defaulted loan exposure identified and the improvement over random review. The aim was to find a practical review strategy rather than build an automatic approval or rejection system.


In [0]:
import math
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.functions import vector_to_array
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.evaluation import BinaryClassificationEvaluator

### 1. Loading the data

I loaded the original Lending Club CSV file from a Databricks volume. I first kept the data unchanged so that I could check its size, columns and overall quality before making any cleaning decisions.


In [0]:
file_path = "/Volumes/workspace/credit_risk/raw_data/loan.csv"
raw_df = spark.read.option("header", True).option("inferSchema", True).option("multiLine", True).option("escape", '"').csv(file_path)
print("Total rows:", raw_df.count())
print("Total columns:", len(raw_df.columns))
display(raw_df.limit(5))

Total rows: 2260668
Total columns: 145


id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
null,null,2500,2500,2500.0,36 months,13.56,84.92,C,C1,Chef,10+ years,RENT,55000.0,Not Verified,Dec-2018,Current,n,null,null,debt_consolidation,Debt consolidation,109xx,NY,18.24,0,Apr-2001,1,null,45,9,1,4341,10.3,34,w,2386.02,2386.02,167.02,167.02,113.98,53.04,0.0,0.0,0.0,Feb-2019,84.92,Mar-2019,Feb-2019,0,null,1,Individual,null,null,null,0,0,16901,2,2,1,2,2,12560,69,2,7,2137,28,42000,1,11,2,9,1878,34360,5.9,0,0,140,212,1,1,0,1,null,2,null,0,2,5,3,3,16,7,18,5,9,0,0,0,3,100.0,0.0,1,0,60124,16901,36500,18124,null,null,null,null,null,null,null,null,null,null,null,N,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Cash,N,null,null,null,null,null,null
null,null,30000,30000,30000.0,60 months,18.94,777.23,D,D2,Postmaster,10+ years,MORTGAGE,90000.0,Source Verified,Dec-2018,Current,n,null,null,debt_consolidation,Debt consolidation,713xx,LA,26.52,0,Jun-1987,0,71,75,13,1,12315,24.2,44,w,29387.75,29387.75,1507.11,1507.11,612.25,894.86,0.0,0.0,0.0,Feb-2019,777.23,Mar-2019,Feb-2019,0,null,1,Individual,null,null,null,0,1208,321915,4,4,2,3,3,87153,88,4,5,998,57,50800,2,15,2,10,24763,13761,8.3,0,0,163,378,4,3,3,4,null,4,null,0,2,4,4,9,27,8,14,4,13,0,0,0,6,95.0,0.0,1,0,372872,99468,15000,94072,null,null,null,null,null,null,null,null,null,null,null,N,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Cash,N,null,null,null,null,null,null
null,null,5000,5000,5000.0,36 months,17.97,180.69,D,D1,Administrative,6 years,MORTGAGE,59280.0,Source Verified,Dec-2018,Current,n,null,null,debt_consolidation,Debt consolidation,490xx,MI,10.51,0,Apr-2011,0,null,null,8,0,4599,19.1,13,w,4787.21,4787.21,353.89,353.89,212.79,141.1,0.0,0.0,0.0,Feb-2019,180.69,Mar-2019,Feb

### 2. Initial data-quality checks

I checked the number of rows and columns and looked for fully duplicated records. This was important because duplicate loans would make the portfolio totals and default-rate calculations inaccurate.


In [0]:
duplicate_rows = raw_df.count() - raw_df.dropDuplicates().count()
print("Duplicate rows:", duplicate_rows)

Duplicate rows: 0


### 3. Selecting relevant columns

The original file contains 145 columns, but many were not needed for this analysis. I kept the variables required for portfolio reporting, historical default analysis and model development.

Payment and recovery fields were retained for portfolio reporting only. I did not use them as model inputs because they are recorded after a loan has been issued and would cause data leakage.


In [0]:
selected_columns = [
    "loan_amnt", "funded_amnt", "term", "int_rate", "installment",
    "grade", "sub_grade", "emp_length", "home_ownership", "annual_inc",
    "verification_status", "issue_d", "loan_status", "purpose", "addr_state",
    "dti", "delinq_2yrs", "earliest_cr_line", "inq_last_6mths", "open_acc",
    "pub_rec", "revol_bal", "revol_util", "total_acc", "application_type",
    "out_prncp", "total_pymnt", "total_rec_prncp", "total_rec_int", "recoveries"
]

loan_df = raw_df.select(selected_columns)

print("Selected columns:", len(loan_df.columns))
display(loan_df.limit(5))

Selected columns: 30


loan_amnt,funded_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,addr_state,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,application_type,out_prncp,total_pymnt,total_rec_prncp,total_rec_int,recoveries
2500,2500,36 months,13.56,84.92,C,C1,10+ years,RENT,55000.0,Not Verified,Dec-2018,Current,debt_consolidation,NY,18.24,0,Apr-2001,1,9,1,4341,10.3,34,Individual,2386.02,167.02,113.98,53.04,0.0
30000,30000,60 months,18.94,777.23,D,D2,10+ years,MORTGAGE,90000.0,Source Verified,Dec-2018,Current,debt_consolidation,LA,26.52,0,Jun-1987,0,13,1,12315,24.2,44,Individual,29387.75,1507.11,612.25,894.86,0.0
5000,5000,36 months,17.97,180.69,D,D1,6 years,MORTGAGE,59280.0,Source Verified,Dec-2018,Current,debt_consolidation,MI,10.51,0,Apr-2011,0,8,0,4599,19.1,13,Individual,4787.21,353.89,212.79,141.1,0.0
4000,4000,36 months,18.94,146.51,D,D2,10+ years,MORTGAGE,92000.0,Source Verified,Dec-2018,Current,debt_consolidation,WA,16.74,0,Feb-2006,0,10,0,5468,78.1,13,Individual,3831.93,286.71,168.07,118.64,0.0
30000,30000,60 months,16.14,731.78,C,C4,10+ years,MORTGAGE,57250.0,Not Verified,Dec-2018,Current,debt_consolidation,MD,26.35,0,Dec-2000,0,12,0,829,3.6,26,Individual,29339.02,1423.21,660.98,762.23,0.0


#### Checking loan status

I reviewed the available loan-status values to understand how the loans were distributed and to check for incorrectly imported records. This also helped me separate active loans from loans with a known final outcome.


In [0]:
status_summary = loan_df.groupBy("loan_status").count().orderBy(F.desc("count"))
status_summary.show(truncate=False)

+---------------------------------------------------+-------+
|loan_status                                        |count  |
+---------------------------------------------------+-------+
|Fully Paid                                         |1041952|
|Current                                            |919695 |
|Charged Off                                        |261655 |
|Late (31-120 days)                                 |21897  |
|In Grace Period                                    |8952   |
|Late (16-30 days)                                  |3737   |
|Does not meet the credit policy. Status:Fully Paid |1988   |
|Does not meet the credit policy. Status:Charged Off|761    |
|Default                                            |31     |
+---------------------------------------------------+-------+



### 4. Creating the analysis populations

I created separate datasets for portfolio reporting and model development because they answer different questions.

The portfolio dataset includes every loan with a valid status and is used to measure volume, exposure and current loan status. The completed-loan dataset includes only `Fully Paid`, `Charged Off` and `Default` loans because their outcomes are known.

Loans marked as not meeting the older credit policy were kept in the portfolio totals but excluded from modelling because they came from a different lending-policy group.


In [0]:
valid_statuses = ["Fully Paid", "Current", "Charged Off", "Late (31-120 days)", "In Grace Period", "Late (16-30 days)", "Does not meet the credit policy. Status:Fully Paid", "Does not meet the credit policy. Status:Charged Off", "Default"]

invalid_rows = loan_df.filter(F.col("loan_status").isNull() | ~F.col("loan_status").isin(valid_statuses)).count()
print("Invalid loan-status rows:", invalid_rows)

portfolio_df = loan_df.filter(F.col("loan_status").isin(valid_statuses))
completed_df = portfolio_df.filter(F.col("loan_status").isin(["Fully Paid", "Charged Off", "Default"]))

completed_df = completed_df.withColumn("default_flag", F.when(F.col("loan_status").isin(["Charged Off", "Default"]), 1).otherwise(0))

print("Portfolio rows:", portfolio_df.count())
print("Completed-loan rows:", completed_df.count())
completed_df.groupBy("default_flag").count().orderBy("default_flag").show()

Invalid loan-status rows: 0
Portfolio rows: 2260668
Completed-loan rows: 1303638
+------------+-------+
|default_flag|  count|
+------------+-------+
|           0|1041952|
|           1| 261686|
+------------+-------+



### 5. Data cleaning and feature preparation

I checked the data types and missing values before starting the analysis. Correct data types were required for calculations, date-based analysis and model training.


In [0]:
completed_df.printSchema()

root
 |-- loan_amnt: integer (nullable = true)
 |-- funded_amnt: integer (nullable = true)
 |-- term: string (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: double (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- issue_d: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- dti: double (nullable = true)
 |-- delinq_2yrs: integer (nullable = true)
 |-- earliest_cr_line: string (nullable = true)
 |-- inq_last_6mths: integer (nullable = true)
 |-- open_acc: integer (nullable = true)
 |-- pub_rec: integer (nullable = true)
 |-- revol_bal: integer (nullable = true)
 |-- revol_util: double (nullable = true)
 |-- total_acc: integer (null

#### Checking missing values

I calculated the number and percentage of missing values in each selected column. I used this check to decide whether missing values should be filled, recorded as an unknown category or excluded.


In [0]:
missing_counts = completed_df.select([F.sum(F.col(column).isNull().cast("int")).alias(column) for column in completed_df.columns]).first().asDict()

total_completed_rows = completed_df.count()

missing_data = [(column, count, round(count / total_completed_rows * 100, 2)) for column, count in missing_counts.items() if count > 0]

missing_df = spark.createDataFrame(missing_data, ["column", "missing_count", "missing_percentage"])
display(missing_df.orderBy(F.desc("missing_percentage")))

column,missing_count,missing_percentage
revol_util,810,0.06
dti,312,0.02
inq_last_6mths,1,0.0


In [0]:
completed_df.groupBy("emp_length").count().orderBy(F.desc("count")).show(20, truncate=False)

+----------+------+
|emp_length|count |
+----------+------+
|10+ years |428553|
|2 years   |117825|
|< 1 year  |104552|
|3 years   |104204|
|1 year    |85678 |
|5 years   |81623 |
|4 years   |78033 |
|n/a       |75457 |
|6 years   |60934 |
|8 years   |59127 |
|7 years   |58148 |
|9 years   |49504 |
+----------+------+



#### Handling missing values

Missing employment length was shown as `n/a`. I changed this to `Unknown` because estimating a borrower’s employment history would not be reliable.

Only a small number of values were missing from DTI, revolving utilisation and recent enquiries. I filled these using the median of each column because the median is less affected by extreme values than the mean.


In [0]:
clean_completed_df = completed_df.withColumn("emp_length", F.when(F.col("emp_length") == "n/a", "Unknown").otherwise(F.col("emp_length")))
dti_median = clean_completed_df.select(F.percentile_approx("dti", 0.5)).first()[0]
revol_util_median = clean_completed_df.select(F.percentile_approx("revol_util", 0.5)).first()[0]
inq_median = clean_completed_df.select(F.percentile_approx("inq_last_6mths", 0.5)).first()[0]

In [0]:
print("DTI median:", dti_median)
print("Revolving utilisation median:", revol_util_median)
print("Recent enquiries median:", inq_median)

DTI median: 17.61
Revolving utilisation median: 52.3
Recent enquiries median: 0


In [0]:
clean_completed_df = clean_completed_df.fillna({"dti": dti_median, "revol_util": revol_util_median, "inq_last_6mths": inq_median})

clean_completed_df.select(
    F.sum(F.col("dti").isNull().cast("int")).alias("missing_dti"),
    F.sum(F.col("revol_util").isNull().cast("int")).alias("missing_revol_util"),
    F.sum(F.col("inq_last_6mths").isNull().cast("int")).alias("missing_enquiries")
).show()

+-----------+------------------+-----------------+
|missing_dti|missing_revol_util|missing_enquiries|
+-----------+------------------+-----------------+
|          0|                 0|                0|
+-----------+------------------+-----------------+



In [0]:
clean_completed_df = clean_completed_df.withColumn("term_months", F.regexp_extract("term", r"\d+", 0).cast("int"))

clean_completed_df = clean_completed_df.withColumn("issue_date", F.to_date("issue_d", "MMM-yyyy"))

clean_completed_df = clean_completed_df.withColumn("earliest_credit_date", F.to_date("earliest_cr_line", "MMM-yyyy"))

clean_completed_df = clean_completed_df.withColumn("issue_year", F.year("issue_date"))

clean_completed_df = clean_completed_df.withColumn("credit_history_years", F.round(F.months_between("issue_date", "earliest_credit_date") / 12, 1))


clean_completed_df.select("term", "term_months", "issue_d", "issue_date", "issue_year", "earliest_cr_line", "earliest_credit_date", "credit_history_years").show(5, truncate=False)

+----------+-----------+--------+----------+----------+----------------+--------------------+--------------------+
|term      |term_months|issue_d |issue_date|issue_year|earliest_cr_line|earliest_credit_date|credit_history_years|
+----------+-----------+--------+----------+----------+----------------+--------------------+--------------------+
| 36 months|36         |Dec-2018|2018-12-01|2018      |Jan-2012        |2012-01-01          |6.9                 |
| 60 months|60         |Dec-2018|2018-12-01|2018      |Jun-2009        |2009-06-01          |9.5                 |
| 36 months|36         |Dec-2018|2018-12-01|2018      |Feb-1999        |1999-02-01          |19.8                |
| 36 months|36         |Dec-2018|2018-12-01|2018      |Dec-2003        |2003-12-01          |15.0                |
| 36 months|36         |Dec-2018|2018-12-01|2018      |Oct-1997        |1997-10-01          |21.2                |
+----------+-----------+--------+----------+----------+----------------+--------

#### Checking loan maturity

The data ends in December 2018, so many recently issued loans were still active at that point. These loans had not been observed for long enough to reach a final outcome.

I calculated the completion rate for each issue year to see how strongly this affected the recent years. This check was necessary before choosing the loans used for model training.


In [0]:
portfolio_prepared_df = portfolio_df.withColumn("issue_date", F.to_date("issue_d", "MMM-yyyy"))

portfolio_prepared_df = portfolio_prepared_df.withColumn("issue_year", F.year("issue_date"))

year_summary = portfolio_prepared_df.groupBy("issue_year").agg(
    F.count("*").alias("total_loans"),
    F.sum(F.when(F.col("loan_status").isin(["Fully Paid", "Charged Off", "Default"]), 1).otherwise(0)).alias("completed_loans")
)
year_summary = year_summary.withColumn("completion_rate", F.round(F.col("completed_loans") / F.col("total_loans") * 100, 2))

year_summary.orderBy("issue_year").show(20, truncate=False)

+----------+-----------+---------------+---------------+
|issue_year|total_loans|completed_loans|completion_rate|
+----------+-----------+---------------+---------------+
|2007      |603        |251            |41.63          |
|2008      |2393       |1562           |65.27          |
|2009      |5281       |4716           |89.3           |
|2010      |12537      |11536          |92.02          |
|2011      |21721      |21721          |100.0          |
|2012      |53367      |53367          |100.0          |
|2013      |134814     |134793         |99.98          |
|2014      |235629     |221468         |93.99          |
|2015      |421095     |373412         |88.68          |
|2016      |434407     |274711         |63.24          |
|2017      |443579     |158910         |35.82          |
|2018      |495242     |47191          |9.53           |
+----------+-----------+---------------+---------------+



#### Selecting mature loans for modelling

For modelling, I kept loans whose scheduled term had finished before the dataset ended. I estimated the completion date using the issue date and loan term, and added a three-month outcome window.

This rule reduces the risk of treating very recent completed loans as representative of all recent lending. The resulting mature-loan group provides a more reliable historical sample for training and testing the model.


In [0]:
data_cutoff_date = F.to_date(F.lit("2018-12-01"))
clean_completed_df = clean_completed_df.withColumn("expected_completion_date", F.add_months(F.col("issue_date"), F.col("term_months") + 3))
model_df = clean_completed_df.filter(F.col("expected_completion_date") <= data_cutoff_date)
print("All completed loans:", clean_completed_df.count())
print("Mature completed loans:", model_df.count())

All completed loans: 1303638
Mature completed loans: 572994


In [0]:
model_df.groupBy("default_flag").count().orderBy("default_flag").show()
model_df.agg(F.round(F.avg("default_flag") * 100, 2).alias("mature_default_rate")).show()

+------------+------+
|default_flag| count|
+------------+------+
|           0|489323|
|           1| 83671|
+------------+------+

+-------------------+
|mature_default_rate|
+-------------------+
|               14.6|
+-------------------+



### 6. Preparing the portfolio dataset

I retained all valid loans for portfolio reporting, including active and recent loans. I converted the term and issue date into useful formats and combined the detailed loan statuses into broader business groups.

These loans were kept because current and delinquent balances are needed to understand the portfolio’s exposure, even though they cannot be used as labelled outcomes for model training.


In [0]:
portfolio_analysis_df = portfolio_df.withColumn("emp_length", F.when(F.col("emp_length") == "n/a", "Unknown").otherwise(F.col("emp_length")))
portfolio_analysis_df = portfolio_analysis_df.withColumn("term_months", F.regexp_extract("term", r"\d+", 0).cast("int"))
portfolio_analysis_df = portfolio_analysis_df.withColumn("issue_date", F.to_date("issue_d", "MMM-yyyy"))
portfolio_analysis_df = portfolio_analysis_df.withColumn("issue_year", F.year("issue_date"))

portfolio_analysis_df = portfolio_analysis_df.withColumn(
    "status_group",
    F.when(F.col("loan_status") == "Fully Paid", "Fully Paid")
     .when(F.col("loan_status").isin(["Charged Off", "Default"]), "Defaulted")
     .when(F.col("loan_status") == "Current", "Current")
     .when(F.col("loan_status").isin(["Late (16-30 days)", "Late (31-120 days)", "In Grace Period"]), "Delinquent")
     .otherwise("Legacy Policy")
)

portfolio_analysis_df.groupBy("status_group").count().orderBy(F.desc("count")).show()

+-------------+-------+
| status_group|  count|
+-------------+-------+
|   Fully Paid|1041952|
|      Current| 919695|
|    Defaulted| 261686|
|   Delinquent|  34586|
|Legacy Policy|   2749|
+-------------+-------+



### 7. Saving the prepared datasets

I saved the cleaned data as two Delta tables. The first contains the complete loan portfolio, while the second contains mature loans with known outcomes.

Saving the tables avoids repeating the full CSV-cleaning process and makes it possible to use the same prepared data in both PySpark and Spark SQL.


In [0]:
portfolio_analysis_df.write.format("delta").mode("overwrite").saveAsTable("workspace.credit_risk.loan_portfolio")
model_df.write.format("delta").mode("overwrite").saveAsTable("workspace.credit_risk.mature_loan_outcomes")
spark.sql("SHOW TABLES IN workspace.credit_risk").show(truncate=False)
spark.sql("""
SELECT 'loan_portfolio' AS table_name, COUNT(*) AS row_count
FROM workspace.credit_risk.loan_portfolio

UNION ALL

SELECT 'mature_loan_outcomes' AS table_name, COUNT(*) AS row_count
FROM workspace.credit_risk.mature_loan_outcomes
""").show()

+-----------+--------------------+-----------+
|database   |tableName           |isTemporary|
+-----------+--------------------+-----------+
|credit_risk|loan_portfolio      |false      |
|credit_risk|mature_loan_outcomes|false      |
+-----------+--------------------+-----------+

+--------------------+---------+
|          table_name|row_count|
+--------------------+---------+
|      loan_portfolio|  2260668|
|mature_loan_outcomes|   572994|
+--------------------+---------+



### Question 1: What is the size and current risk position of the portfolio?

I started by calculating the main portfolio measures: loan count, total funded amount, outstanding principal, current loans and delinquent exposure.

In this project, delinquent exposure means the outstanding principal linked to loans that were late or in a grace period when the data was collected. It represents an amount that may require attention, not a confirmed loss.


In [0]:
%sql

SELECT
    COUNT(*) AS total_loans,
    ROUND(SUM(funded_amnt), 2) AS total_funded_amount,
    ROUND(AVG(funded_amnt), 2) AS average_funded_amount,
    ROUND(AVG(int_rate), 2) AS average_interest_rate,
    ROUND(SUM(out_prncp), 2) AS outstanding_principal,
    SUM(CASE WHEN status_group = 'Current' THEN 1 ELSE 0 END) AS current_loans,
    ROUND(100.0 * SUM(CASE WHEN status_group = 'Current' THEN 1 ELSE 0 END) / COUNT(*), 2) AS current_loan_percentage,
    SUM(CASE WHEN status_group = 'Delinquent' THEN 1 ELSE 0 END) AS delinquent_loans,
    ROUND(100.0 * SUM(CASE WHEN status_group = 'Delinquent' THEN 1 ELSE 0 END) / COUNT(*), 2) AS delinquent_loan_percentage,
    ROUND(SUM(CASE WHEN status_group = 'Delinquent' THEN out_prncp ELSE 0 END), 2) AS delinquent_exposure,
    ROUND(100.0 * SUM(CASE WHEN status_group = 'Delinquent' THEN out_prncp ELSE 0 END) / SUM(out_prncp), 2) AS delinquent_exposure_percentage
FROM workspace.credit_risk.loan_portfolio;

total_loans,total_funded_amount,average_funded_amount,average_interest_rate,outstanding_principal,current_loans,current_loan_percentage,delinquent_loans,delinquent_loan_percentage,delinquent_exposure,delinquent_exposure_percentage
2260668,34004208600,15041.66,13.09,1.005159204029E10,919695,40.68,34586,1.53,3.8539812915E8,3.83


#### Finding

The dataset contains 2.26 million loans with a total funded value of about $34.00 billion. Approximately $10.05 billion of principal was still outstanding at the end of the dataset.

There were 34,586 delinquent loans, equal to about 1.53% of all portfolio loans. Their outstanding exposure was approximately $385.40 million, or 3.83% of total outstanding principal.

The exposure share was higher than the share of delinquent loans. This suggests that delinquent accounts had larger outstanding balances on average, so I examined the portfolio further by grade, term and purpose.


### Question 2: Which grades combine higher default risk with substantial exposure?

Lending Club grades range from A to G, where A generally represents lower assessed risk and G represents higher assessed risk.

I compared historical default rates from mature loans with outstanding principal in the complete portfolio. This comparison shows whether a grade is important because of its default rate, its financial exposure or both.


In [0]:
%sql

WITH portfolio_by_grade AS (
    SELECT
        grade,
        COUNT(*) AS portfolio_loans,
        ROUND(SUM(out_prncp), 2) AS outstanding_principal,
        ROUND(SUM(CASE WHEN status_group = 'Delinquent' THEN out_prncp ELSE 0 END), 2) AS delinquent_exposure
    FROM workspace.credit_risk.loan_portfolio
    WHERE grade IS NOT NULL
    GROUP BY grade
),

outcomes_by_grade AS (
    SELECT
        grade,
        COUNT(*) AS mature_loans,
        SUM(default_flag) AS defaulted_loans,
        ROUND(AVG(default_flag) * 100, 2) AS historical_default_rate
    FROM workspace.credit_risk.mature_loan_outcomes
    WHERE grade IS NOT NULL
    GROUP BY grade
)

SELECT
    p.grade,
    p.portfolio_loans,
    o.mature_loans,
    o.defaulted_loans,
    o.historical_default_rate,
    p.outstanding_principal,
    p.delinquent_exposure
FROM portfolio_by_grade p
LEFT JOIN outcomes_by_grade o
    ON p.grade = o.grade
ORDER BY p.grade;

grade,portfolio_loans,mature_loans,defaulted_loans,historical_default_rate,outstanding_principal,delinquent_exposure
A,433027,122243,6769,5.54,2.11882516286E9,2.177409219E7
B,663557,186519,20905,11.21,2.82465638583E9,7.362442336E7
C,650053,151744,27225,17.94,3.00423870137E9,1.3425654029E8
D,324424,74721,17344,23.21,1.44721345799E9,9.201131577E7
E,135639,27472,7879,28.68,4.8456431448E8,4.200237595E7
F,41800,8781,2987,34.02,1.2888496669E8,1.543791345E7
G,12168,1514,562,37.12,4.320905107E7,6291468.14


#### Finding

Historical default rates increased steadily from Grade A to Grade G. Grade A had the lowest rate at 5.54%, while Grade G had the highest at 37.12%.

Grade G, however, had only about $43.21 million in outstanding principal. Grade C had the largest outstanding principal at approximately $3.00 billion and also had the highest number of historical defaults, at 27,225.

Grades C and D together represented about $4.45 billion in outstanding principal. For this reason, review activity should not focus only on Grades F and G. The larger size of Grades C and D means they may create a greater overall financial impact despite having lower default rates.


### Question 3: Does loan term affect default risk within each grade?

Loan grade may not explain all of the risk in a loan. A longer repayment term keeps the borrower in debt for more time and may introduce additional uncertainty.

I compared 36-month and 60-month loans within each grade to check whether term provides useful information beyond the grade itself.


In [0]:
%sql

WITH segment_summary AS (
    SELECT
        grade,
        term_months,
        COUNT(*) AS mature_loans,
        SUM(default_flag) AS defaulted_loans,
        ROUND(AVG(default_flag) * 100, 2) AS historical_default_rate,
        ROUND(AVG(funded_amnt), 2) AS average_funded_amount,
        ROUND(SUM(CASE WHEN default_flag = 1 THEN funded_amnt ELSE 0 END), 2) AS defaulted_funded_exposure
    FROM workspace.credit_risk.mature_loan_outcomes
    WHERE grade IS NOT NULL
      AND term_months IS NOT NULL
    GROUP BY grade, term_months
)

SELECT
    grade,
    term_months,
    mature_loans,
    defaulted_loans,
    historical_default_rate,
    average_funded_amount,
    defaulted_funded_exposure,
    DENSE_RANK() OVER (ORDER BY historical_default_rate DESC) AS risk_rank
FROM segment_summary
ORDER BY grade, term_months;

grade,term_months,mature_loans,defaulted_loans,historical_default_rate,average_funded_amount,defaulted_funded_exposure,risk_rank
A,36,121135,6677,5.51,13751.18,87356275,14
A,60,1108,92,8.3,14395.8,1365900,13
B,36,179544,19800,11.03,12404.26,240026900,12
B,60,6975,1105,15.84,18166.22,19462100,11
C,36,138348,24263,17.54,11836.05,285218025,10
C,60,13396,2962,22.11,19179.07,55910075,9
D,36,67444,15369,22.79,11759.75,185687950,8
D,60,7277,1975,27.14,19282.04,37741150,7
E,36,19135,5315,27.78,12005.17,66576550,6
E,60,8337,2564,30.75,21157.3,54172525,5


#### Finding

Within every grade, 60-month loans had a higher historical default rate than 36-month loans. This indicates that term adds useful information when assessing risk.

For Grade C, the default rate increased from 17.54% for 36-month loans to 22.11% for 60-month loans. For Grade D, it increased from 22.79% to 27.14%.

Longer-term loans also tended to have higher average funded amounts. Grade C and D loans with 60-month terms may therefore deserve additional attention because they combine meaningful portfolio volume, larger balances and higher default rates.


### Question 4: Which loan purposes create the greatest risk and exposure?

A purpose category can have a high default rate but contain very few loans. Another category may have a lower rate but create more total exposure because it is much larger.

I therefore compared both historical default rates and defaulted funded exposure instead of ranking purposes using default rate alone.


In [0]:
%sql

WITH portfolio_purpose AS (
    SELECT
        purpose,
        COUNT(*) AS portfolio_loans,
        ROUND(SUM(out_prncp), 2) AS outstanding_principal,
        ROUND(SUM(CASE WHEN status_group = 'Delinquent' THEN out_prncp ELSE 0 END), 2) AS delinquent_exposure
    FROM workspace.credit_risk.loan_portfolio
    GROUP BY purpose
),

outcome_purpose AS (
    SELECT
        purpose,
        COUNT(*) AS mature_loans,
        ROUND(AVG(default_flag) * 100, 2) AS historical_default_rate,
        ROUND(SUM(CASE WHEN default_flag = 1 THEN funded_amnt ELSE 0 END), 2) AS defaulted_funded_exposure
    FROM workspace.credit_risk.mature_loan_outcomes
    GROUP BY purpose
)

SELECT
    o.purpose,
    o.mature_loans,
    o.historical_default_rate,
    o.defaulted_funded_exposure,
    p.portfolio_loans,
    p.outstanding_principal,
    p.delinquent_exposure
FROM outcome_purpose o
LEFT JOIN portfolio_purpose p
    ON o.purpose = p.purpose
ORDER BY o.defaulted_funded_exposure DESC;

purpose,mature_loans,historical_default_rate,defaulted_funded_exposure,portfolio_loans,outstanding_principal,delinquent_exposure
debt_consolidation,330076,15.37,697332250,1277877,5.87229795785E9,2.3432552889E8
credit_card,133257,12.03,216417425,516971,2.43115642508E9,6.782617238E7
home_improvement,32708,12.76,55474425,150457,6.5889471211E8,2.720511516E7
other,30633,16.91,46508675,139440,4.7930082197E8,2.410930794E7
small_business,7702,24.47,28960475,24689,1.1039457811E8,7003405.58
major_purchase,12069,12.85,16697550,50445,2.0049286441E8,1.071427126E7
medical,5999,16.97,8151400,27488,8.243598377E7,3842507.21
car,6842,11.52,6650075,24013,6.37389212E7,2472587.11
house,2662,17.66,6210900,14136,8.351117779E7,4303227.75
moving,4179,19.17,5905150,15403,3.725282935E7,2053565.39


#### Finding

Small-business loans had the highest historical default rate among the main purpose categories, at 24.47%. Their historical defaulted funded exposure was approximately $28.96 million.

Debt-consolidation loans had a lower default rate of 15.37%, but the portfolio contained more than 1.27 million of these loans. Their historical defaulted funded exposure was approximately $697.33 million. Credit-card loans were second at approximately $216.42 million.

This result shows the difference between individual risk and portfolio impact. Small-business loans were riskier by rate, while debt-consolidation loans created the greatest total exposure because they were much more common.


### Question 5: How did lending activity change over time?

I analysed annual loan count, funded amount, average loan size and average interest rate to understand how the Lending Club portfolio developed between 2007 and 2018.

I also calculated the annual percentage change in funded amount to identify periods of rapid growth and slower activity.


In [0]:
%sql

WITH yearly_lending AS (
    SELECT
        issue_year,
        COUNT(*) AS loan_count,
        SUM(funded_amnt) AS total_funded_amount,
        AVG(funded_amnt) AS average_funded_amount,
        AVG(int_rate) AS average_interest_rate
    FROM workspace.credit_risk.loan_portfolio
    WHERE issue_year IS NOT NULL
    GROUP BY issue_year
)

SELECT
    issue_year,
    loan_count,
    ROUND(total_funded_amount, 2) AS total_funded_amount,
    ROUND(average_funded_amount, 2) AS average_funded_amount,
    ROUND(average_interest_rate, 2) AS average_interest_rate,
    ROUND(
        (total_funded_amount - LAG(total_funded_amount) OVER (ORDER BY issue_year))
        / LAG(total_funded_amount) OVER (ORDER BY issue_year) * 100,
        2
    ) AS funded_amount_growth_percentage
FROM yearly_lending
ORDER BY issue_year;

issue_year,loan_count,total_funded_amount,average_funded_amount,average_interest_rate,funded_amount_growth_percentage
2007,603,4791550,7946.19,11.83,null
2008,2393,19975025,8347.27,12.06,316.88
2009,5281,51814750,9811.54,12.44,159.4
2010,12537,126351175,10078.26,11.99,143.85
2011,21721,257363650,11848.61,12.22,103.69
2012,53367,717942625,13452.93,13.64,178.96
2013,134814,1982759550,14707.37,14.53,176.17
2014,235629,3503840175,14870.16,13.77,76.72
2015,421095,6417608175,15240.29,12.6,83.16
2016,434407,6400541700,14733.97,13.04,-0.27


#### Finding

Lending activity grew strongly between 2007 and 2015. Annual funded value increased from approximately $4.79 million in 2007 to $6.42 billion in 2015.

Growth slowed after 2015. Funded value fell slightly by 0.27% in 2016 and increased by 2.88% in 2017. It then rose by 20.52% in 2018, reaching approximately $7.94 billion.

Average funded amount also increased, from about $7,946 in 2007 to $16,025 in 2018. This suggests that the platform grew through both a higher number of loans and larger average loan values. The very high early growth rates should be treated carefully because they were calculated from a small starting portfolio.


### Question 6: How is borrower debt burden related to default risk?

The debt-to-income ratio (DTI) compares a borrower’s monthly debt obligations with income. A higher DTI can indicate that less income is available for additional repayments.

I grouped mature loans into DTI bands and compared their historical default rates and defaulted funded exposure.


In [0]:
%sql

WITH dti_segments AS (
    SELECT
        CASE
            WHEN dti < 0 THEN 'Invalid'
            WHEN dti < 10 THEN 'Under 10'
            WHEN dti < 20 THEN '10-19.99'
            WHEN dti < 30 THEN '20-29.99'
            WHEN dti < 40 THEN '30-39.99'
            ELSE '40 or more'
        END AS dti_band,
        CASE
            WHEN dti < 0 THEN 0
            WHEN dti < 10 THEN 1
            WHEN dti < 20 THEN 2
            WHEN dti < 30 THEN 3
            WHEN dti < 40 THEN 4
            ELSE 5
        END AS band_order,
        default_flag,
        funded_amnt
    FROM workspace.credit_risk.mature_loan_outcomes
)

SELECT
    dti_band,
    COUNT(*) AS mature_loans,
    SUM(default_flag) AS defaulted_loans,
    ROUND(AVG(default_flag) * 100, 2) AS historical_default_rate,
    ROUND(AVG(funded_amnt), 2) AS average_funded_amount,
    ROUND(SUM(CASE WHEN default_flag = 1 THEN funded_amnt ELSE 0 END), 2) AS defaulted_funded_exposure
FROM dti_segments
GROUP BY dti_band, band_order
ORDER BY band_order;

dti_band,mature_loans,defaulted_loans,historical_default_rate,average_funded_amount,defaulted_funded_exposure
Under 10,113988,12902,11.32,12323.91,162630000
10-19.99,249335,33608,13.48,13345.47,452295875
20-29.99,168405,28771,17.08,13198.48,381967175
30-39.99,41266,8390,20.33,11898.9,99875000


#### Finding

Historical default rates increased with DTI. Loans with a DTI below 10 had a default rate of 11.32%, compared with 20.33% for loans in the 30–39.99 band.

The 10–19.99 band produced the largest defaulted funded exposure, at approximately $452.30 million, because it contained the most loans. The 20–29.99 band followed at approximately $381.97 million.

High-DTI borrowers showed greater individual risk, but the middle DTI groups created more total exposure because of their size. DTI should therefore be considered together with loan amount, grade and term when deciding which loans to review.


## 8. Loan-review prioritisation

The earlier analysis showed separate patterns across grade, term, purpose and DTI. A review decision, however, needs to consider several loan and borrower characteristics at the same time.

I used Logistic Regression to estimate a default-risk probability for each mature loan. The purpose of the model was to test whether a limited review team could find more historical defaults by examining higher-ranked loans first. It was not designed to make automatic lending decisions.

I used only information available at or near the time of underwriting. Payment, recovery and final-outcome fields were excluded to avoid data leakage.


In [0]:
numeric_features = ["loan_amnt", "term_months", "int_rate", "installment", "annual_inc", "dti", "delinq_2yrs", "inq_last_6mths", "open_acc", "pub_rec", "revol_bal", "revol_util", "total_acc", "credit_history_years"]

categorical_features = ["grade", "emp_length", "home_ownership", "verification_status", "purpose", "application_type"]
model_columns = numeric_features + categorical_features + ["default_flag", "funded_amnt", "issue_date"]
model_data = model_df.select(model_columns)
print("Model rows:", model_data.count())
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Model rows: 572994
Numeric features: 14
Categorical features: 6


#### Training and testing data

I split the mature-loan data into 80% for training and 20% for testing. I used a fixed random seed so that the same split can be reproduced.

The model learned from the training set only. The test set was kept separate and used to measure performance on unseen records.

Logistic Regression requires numerical inputs. I converted categorical variables using `StringIndexer` and one-hot encoding, and then combined them with the numerical variables in a single feature vector.


In [0]:
training_data, testing_data = model_data.randomSplit([0.8, 0.2], seed=42)
print("Training rows:", training_data.count())
print("Testing rows:", testing_data.count())
indexers = [StringIndexer(inputCol=column, outputCol=column + "_index", handleInvalid="keep") for column in categorical_features]
indexed_columns = [column + "_index" for column in categorical_features]
encoded_columns = [column + "_encoded" for column in categorical_features]
encoder = OneHotEncoder(inputCols=indexed_columns, outputCols=encoded_columns)
assembler = VectorAssembler(inputCols=numeric_features + encoded_columns, outputCol="features", handleInvalid="keep")

Training rows: 458456
Testing rows: 114538


In [0]:
logistic_regression = LogisticRegression(featuresCol="features", labelCol="default_flag", probabilityCol="probability", maxIter=50)
model_pipeline = Pipeline(stages=indexers + [encoder, assembler, logistic_regression])

#### Model training

I fitted the complete preprocessing and Logistic Regression pipeline using the training data. The fitted pipeline then applied the same preparation steps to the test data.

For each test loan, the model returned a probability between 0 and 1. A larger value indicates a higher estimated probability of historical default.


In [0]:
fitted_model = model_pipeline.fit(training_data)
predictions = fitted_model.transform(testing_data)
predictions = predictions.withColumn("risk_probability", vector_to_array("probability")[1])
predictions.select("default_flag", "prediction", "risk_probability", "funded_amnt").show(10, truncate=False)
roc_evaluator = BinaryClassificationEvaluator(labelCol="default_flag", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
pr_evaluator = BinaryClassificationEvaluator(labelCol="default_flag", rawPredictionCol="rawPrediction", metricName="areaUnderPR")
roc_auc = roc_evaluator.evaluate(predictions)
pr_auc = pr_evaluator.evaluate(predictions)
print("ROC-AUC:", round(roc_auc, 3))
print("PR-AUC:", round(pr_auc, 3))

+------------+----------+-------------------+-----------+
|default_flag|prediction|risk_probability   |funded_amnt|
+------------+----------+-------------------+-----------+
|0           |0.0       |0.03972022360497551|800        |
|0           |0.0       |0.11517518141331995|950        |
|0           |0.0       |0.06627481937222346|1000       |
|0           |0.0       |0.05029997204002912|1000       |
|0           |0.0       |0.10008503979776384|1000       |
|0           |0.0       |0.03181689294893164|1000       |
|0           |0.0       |0.05295974043256668|1000       |
|0           |0.0       |0.06111087310035279|1000       |
|1           |0.0       |0.05191485385327821|1000       |
|0           |0.0       |0.0662197112667341 |1000       |
+------------+----------+-------------------+-----------+
only showing top 10 rows
ROC-AUC: 0.679
PR-AUC: 0.255


#### Model performance

The model achieved a ROC-AUC of 0.679. This means it had a moderate ability to rank defaulted loans above fully paid loans.

Its PR-AUC was 0.255, compared with a random baseline of approximately 0.146. The improvement suggests that the model provides useful risk separation, but the results are not strong enough to justify automatic approval or rejection.

I therefore used the model only as a way to prioritise loans for manual review.


### Manual-review capacity analysis

I ranked the test loans from the highest to the lowest predicted default probability. I then simulated teams able to review only 5%, 10% or 20% of the test loans.

For each capacity, I measured review precision, the share of historical defaults captured, lift over random review and the funded amount linked to the captured defaults.


In [0]:
prediction_results = predictions.select("default_flag", "funded_amnt", "risk_probability")
prediction_results.write.format("delta").mode("overwrite").saveAsTable("workspace.credit_risk.lr_test_predictions")

In [0]:
prediction_results = spark.table("workspace.credit_risk.lr_test_predictions")
ranking_window = Window.orderBy(F.desc("risk_probability"))
ranked_predictions = prediction_results.withColumn("risk_rank", F.row_number().over(ranking_window))
ranked_predictions.write.format("delta").mode("overwrite").saveAsTable("workspace.credit_risk.lr_ranked_predictions")
ranked_predictions = spark.table("workspace.credit_risk.lr_ranked_predictions")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
total_test_loans = ranked_predictions.count()
total_test_defaults = ranked_predictions.filter(F.col("default_flag") == 1).count()
total_defaulted_exposure = ranked_predictions.filter(F.col("default_flag") == 1).agg(F.sum("funded_amnt")).first()[0]

print("Test loans:", total_test_loans)
print("Test defaults:", total_test_defaults)
print("Defaulted funded exposure:", total_defaulted_exposure)

Test loans: 114538
Test defaults: 16683
Defaulted funded exposure: 219100550


In [0]:
review_results = []

for capacity in [0.05, 0.10, 0.20]:
    review_limit = math.ceil(total_test_loans * capacity)
    actual_review_rate = review_limit / total_test_loans
    reviewed = ranked_predictions.filter(F.col("risk_rank") <= review_limit)
    defaults_captured = reviewed.filter(F.col("default_flag") == 1).count()
    exposure_captured = reviewed.filter(F.col("default_flag") == 1).agg(F.sum("funded_amnt")).first()[0]
    precision = defaults_captured / review_limit
    capture_rate = defaults_captured / total_test_defaults
    lift = capture_rate / actual_review_rate
    exposure_capture_rate = exposure_captured / total_defaulted_exposure
    review_results.append((int(capacity * 100), review_limit, defaults_captured, round(precision * 100, 2), round(capture_rate * 100, 2), round(lift, 2), float(exposure_captured), round(exposure_capture_rate * 100, 2)))

In [0]:
review_results_df = spark.createDataFrame(review_results, ["review_capacity", "loans_reviewed", "defaults_captured", "precision_percentage", "default_capture_rate", "lift_over_random", "defaulted_exposure_identified", "exposure_capture_rate"])
display(review_results_df.orderBy("review_capacity"))

review_capacity,loans_reviewed,defaults_captured,precision_percentage,default_capture_rate,lift_over_random,defaulted_exposure_identified,exposure_capture_rate
5,5727,1911,33.37,11.45,2.29,3.132165E7,14.3
10,11454,3471,30.3,20.81,2.08,5.33954E7,24.37
20,22908,6118,26.71,36.67,1.83,8.8384925E7,40.34


#### Review-strategy finding

The model placed a larger share of historical defaults near the top of the review queue.

At 5% capacity, the team would review 5,727 loans and identify 1,911 defaults. This captured 11.45% of all defaults in the test set, achieved 2.29× lift over random review and identified approximately $31.32 million in defaulted funded exposure.

At 10% capacity, 11,454 loans would be reviewed and 3,471 defaults identified. Precision was 30.30%, the default capture rate was 20.81% and lift over random review was 2.08×. The identified defaulted exposure was approximately $53.40 million.

At 20% capacity, the strategy captured 36.67% of defaults and approximately $88.38 million in defaulted exposure. Precision decreased to 26.71% as more lower-ranked loans entered the queue, although lift remained above random at 1.83×.

The 10% option provides a practical balance: it more than doubles random-review effectiveness while keeping the number of reviewed loans relatively limited.


### Comparison with random review

To create a simple operational benchmark, I compared the model’s highest-risk 10% with a random 10% sample from the same test data. I used a fixed seed so that the random sample can be reproduced.


In [0]:
random_window = Window.orderBy(F.rand(seed=42))
random_ranked = prediction_results.withColumn("random_rank", F.row_number().over(random_window))
random_ranked.write.format("delta").mode("overwrite").saveAsTable("workspace.credit_risk.random_review_benchmark")
random_ranked = spark.table("workspace.credit_risk.random_review_benchmark")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
random_review_limit = math.ceil(total_test_loans * 0.10)
random_reviewed = random_ranked.filter(F.col("random_rank") <= random_review_limit)
random_defaults_captured = random_reviewed.filter(F.col("default_flag") == 1).count()
random_exposure_identified = random_reviewed.filter(F.col("default_flag") == 1).agg(F.sum("funded_amnt")).first()[0]
random_precision = random_defaults_captured / random_review_limit
random_capture_rate = random_defaults_captured / total_test_defaults

In [0]:
print("Random loans reviewed:", random_review_limit)
print("Random defaults captured:", random_defaults_captured)
print("Random precision:", round(random_precision * 100, 2), "%")
print("Random default capture rate:", round(random_capture_rate * 100, 2), "%")
print("Random defaulted exposure identified: $", round(random_exposure_identified, 2))

Random loans reviewed: 11454
Random defaults captured: 1641
Random precision: 14.33 %
Random default capture rate: 9.84 %
Random defaulted exposure identified: $ 21911025


In [0]:
comparison_data = [
    ("Random review", 10, 11454, 1641, 14.33, 9.84, 21911025.0),
    ("Model-ranked review", 10, 11454, 3471, 30.30, 20.81, 53395400.0)
]
comparison_df = spark.createDataFrame(comparison_data, ["review_strategy", "capacity_percentage", "loans_reviewed", "defaults_captured", "precision_percentage", "default_capture_rate", "defaulted_exposure_identified"])
display(comparison_df)

review_strategy,capacity_percentage,loans_reviewed,defaults_captured,precision_percentage,default_capture_rate,defaulted_exposure_identified
Random review,10,11454,1641,14.33,9.84,2.1911025E7
Model-ranked review,10,11454,3471,30.3,20.81,5.33954E7


#### Finding

At the same 10% capacity, random review identified 1,641 defaults and achieved 14.33% precision. The model-ranked review identified 3,471 defaults with 30.30% precision.

The model therefore found 1,830 additional defaults without increasing the number of loans reviewed. It also identified approximately $31.48 million more defaulted funded exposure.

For this fixed sample, the model found about 2.11 times as many defaults and 2.44 times as much defaulted exposure as random review. A different random seed could change the exact comparison, but the fixed sample gives a clear and reproducible baseline.


### Checking risk by predicted decile

I divided the test loans into ten equal groups based on their predicted default probability. Decile 1 contains the highest-risk 10% and Decile 10 contains the lowest-risk 10%.

This check shows whether actual default rates fall as the model’s predicted risk becomes lower. A consistent pattern would indicate that the model separates higher-risk and lower-risk loans in a useful way.


In [0]:
%sql

WITH risk_deciles AS (
    SELECT
        default_flag,
        funded_amnt,
        risk_probability,
        NTILE(10) OVER (ORDER BY risk_probability DESC) AS risk_decile
    FROM workspace.credit_risk.lr_test_predictions
)

SELECT
    risk_decile,
    COUNT(*) AS loans,
    SUM(default_flag) AS defaulted_loans,
    ROUND(AVG(risk_probability) * 100, 2) AS average_predicted_risk,
    ROUND(AVG(default_flag) * 100, 2) AS actual_default_rate,
    ROUND(SUM(CASE WHEN default_flag = 1 THEN funded_amnt ELSE 0 END), 2) AS defaulted_funded_exposure
FROM risk_deciles
GROUP BY risk_decile
ORDER BY risk_decile;

risk_decile,loans,defaulted_loans,average_predicted_risk,actual_default_rate,defaulted_funded_exposure
1,11454,3471,31.15,30.3,53395400
2,11454,2647,23.14,23.11,34989525
3,11454,2293,19.51,20.02,28102500
4,11454,1982,16.71,17.3,24151300
5,11454,1661,14.26,14.5,19894200
6,11454,1355,12.14,11.83,16952800
7,11454,1154,10.26,10.08,14122525
8,11454,962,8.26,8.4,11781550
9,11453,706,6.07,6.16,9258775
10,11453,452,4.22,3.95,6451975


#### Finding

Actual default rates decreased consistently across the risk deciles. The highest-risk decile had an actual default rate of 30.30%, while the lowest-risk decile had a rate of 3.95%.

The highest-risk group therefore had about 7.67 times the default rate of the lowest-risk group. It contained 3,471 defaults and approximately $53.40 million in defaulted funded exposure.

Average predicted risk was reasonably close to the actual default rate across the deciles. This supports using the model to order a manual-review queue. However, because both defaulted and fully paid loans appear within the groups, the model should support human judgement rather than replace it.
